# M30 — Dissect a Transformer Block

**Objective:** trace data through a transformer block and reconstruct its roles.

M29 already runs one attention head. The useful whole here is a **tiny
transformer block** on a deterministic 4-D sequence:

`x → LN → multi-head attention → residual → LN → FFN → residual → y`

That order is the **declared teaching convention: pre-norm**. Post-norm
is a labeled alternative, **not a universal** law. Heads are
**parallel learned projections**. Nothing is downloaded. LLM training
and inference stay closed (M31–M32).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a shape, a residual identity, a first
diverging checkpoint, or a changed output coordinate.

Heads do not receive human job titles. A residual is `stream + sublayer`.
Do not open a training loop. Softmax still belongs to M29.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M30" / "transformer_block.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M29.attention_core import (
    X_CASH_CONTEXT as M29_CASH,
    causal_additive_mask,
    scaled_dot_product_attention,
    self_attention as m29_self_attention,
)
from missions.M30.transformer_block import (
    BANK_INDEX,
    BLOCK_VERSION,
    CASH_TOKENS,
    CONTEXT_INDEX,
    D_FF,
    D_HEAD,
    D_MODEL,
    GOLDEN_CASH_ATTN_RESIDUAL,
    GOLDEN_CASH_OUTPUT,
    HEAD_INTERPRETATION_LIMIT,
    LN_VECTOR,
    N_HEADS,
    NORM_INTERPRETATION_LIMIT,
    PRE_NORM_DIAGRAM,
    RESIDUAL_INTERPRETATION_LIMIT,
    RESIDUAL_STREAM,
    RESIDUAL_SUBLAYER,
    RESIDUAL_SUM,
    TEACHING_CONVENTION,
    TEACHING_W_O_HALF,
    TRACE_CHECKPOINTS,
    WATER_TOKENS,
    X_CASH_CONTEXT,
    X_WATER_CONTEXT,
    ablate_residual,
    block_with_defect,
    checkpoint_parity,
    feed_forward,
    first_divergence,
    golden_pre_norm_trace,
    independent_pre_norm_compose,
    layer_norm,
    merge_heads,
    multi_head_attention,
    observability_report,
    params_with_activation,
    params_with_output_projection,
    repair_block,
    residual_add,
    split_heads,
    stream_l2,
    teaching_batch,
    teaching_params,
    transformer_block,
)

print("repository root:", ROOT)
print("declared convention:", TEACHING_CONVENTION)
print("version:", BLOCK_VERSION)
print("d_model / n_heads / d_head / d_ff:", D_MODEL, N_HEADS, D_HEAD, D_FF)
print("diagram:\n", PRE_NORM_DIAGRAM)
print("head limit:", HEAD_INTERPRETATION_LIMIT)


## M29 → M30 boundary: one head in, the block out

M29 already projects Q, K, and V, scales dots, masks, softmaxes over
keys, and mixes values. That call is **trusted**. M30 wraps it.

What this mission **opens:** several heads as parallel projections,
split/merge, output projection `W_O`, residual paths, a **declared**
pre-norm convention (post-norm labeled), a position-wise feed-forward,
and named tensor checkpoints through the block.

What stays **deferred:**
- M31 — next-token objective, training loops, contamination
- M32 — decoding, temperature, adaptation policy

An M29 weight is still not intent. A head is still not a job title.


## Frozen teaching fixtures

Declare the useful whole **before** the first residual add.

| Fixture | Value |
| --- | --- |
| Convention | `pre_norm` (`x + MHA(LN(x))`, then `h + FFN(LN(h))`) |
| Alternative | `post_norm`, labeled, same weights |
| Sequences | `river bank cash` vs `river bank water` in 4-D |
| `d_model`, `n_heads`, `d_head`, `d_ff` | 4, 2, 2, 8 |
| First two coordinates | M29's 2-D cash/water subspace |
| Projections (first pass) | identity, so heads are coordinate slices |
| Activation | ReLU |
| Scale / softmax | M29 defaults: `1/sqrt(d_k)`, keys |
| Version | `v06-teaching-block-1` |
| Download | false — numbers live in `transformer_block.py` |

Primary sources: `hf-llm-course` and `karpathy-zero-to-hero` in
`data/source_registry.json`. Skip pretrained transformers here.

The table is authored so residual and norm bugs are visible. It is not
a language-model quality benchmark.


In [ ]:
params = teaching_params()
cash_x = np.asarray(X_CASH_CONTEXT, dtype=float)
water_x = np.asarray(X_WATER_CONTEXT, dtype=float)
print("cash X shape", cash_x.shape)
print("cash X\n", cash_x)
print("water X\n", water_x)
print("named change is index", CONTEXT_INDEX, "only")
print("bank row equal", np.allclose(cash_x[BANK_INDEX], water_x[BANK_INDEX]))
print("first two coords are M29 cash\n", cash_x[:, :2])
print("params d_model/n_heads/d_ff", params.d_model, params.n_heads, params.d_ff)
print("W_Q is identity", np.allclose(params.w_q, np.eye(4)))
print("hand residual", RESIDUAL_STREAM, "+", RESIDUAL_SUBLAYER, "->", RESIDUAL_SUM)
print("LN demo vector", LN_VECTOR)


### One sequence, two subspaces, one named change

Row-batch layout is still M16: one token per row. The first two
features are the M29 cash/water geometry. The last two are a second
subspace. Identity projections will slice those subspaces into two
heads. That is a geometric accident of this fixture, not a reason to
name the heads.

Keep the tensors. Predict the whole block before any LayerNorm.


## Predict before running — useful whole

Timestamp a prediction before `run-whole`.

Declared **pre-norm** block, identity projections, ReLU FFN, no mask.
The only change between the two sequences is the third token vector.

Predict:
- whether bank's **block output** stays the same in cash vs water
- whether the attention residual at bank can stay fixed if the FFN
  still runs
- whether a 4-D output must equal the 2-D M29 bank vector

Do not assign a meaning to either head. A geometric guess from the
table is the point.


In [ ]:
cash = transformer_block(X_CASH_CONTEXT, params, tokens=CASH_TOKENS)
water = transformer_block(X_WATER_CONTEXT, params, tokens=WATER_TOKENS)
batch = transformer_block(teaching_batch(), params)
print("cash shapes", cash.shapes)
print("cash attn residual bank", cash.attn_residual[0, BANK_INDEX])
print("water attn residual bank", water.attn_residual[0, BANK_INDEX])
print("cash output bank", cash.output[0, BANK_INDEX])
print("water output bank", water.output[0, BANK_INDEX])
print("batch outputs bank\n", batch.output[:, BANK_INDEX])
print("report", observability_report(cash))
assert cash.declared_convention == "pre_norm"
assert cash.shapes["output"] == (1, 3, 4)
assert not np.allclose(cash.output[0, BANK_INDEX], water.output[0, BANK_INDEX])
assert np.allclose(batch.output[0], cash.output[0])
assert np.allclose(batch.output[1], water.output[0])
print("context changed the block output; heads are not job titles")


### The block mixed neighbors, then added, then mixed features

Same token identity at `bank`, different third vector, different
block output. Attention mixed **positions**. The FFN then mixed
**features** at each position. Residuals added those sublayers back
onto the stream.

The numbers did not "understand banking." Next, find M29 inside the
first sublayer, before LayerNorm rewrites the input.


## Predict before running — M29 attention inside a head

Timestamp a prediction before `run-m29-sublayer`.

Identity `W_Q/W_K/W_V` on the 4-D cash sequence, **no** LayerNorm.
Head 0 is the first two coordinates — M29's cash geometry.

Predict:
- head 0 bank weights compared with M29 cash (uniform `1/3`?)
- whether head 1 bank weights equal head 0
- whether this raw MHA is what the pre-norm block feeds M29
  (hint: the block LayerNorms first)

The M29 call is composed, not rewritten.


In [ ]:
m29_cash = m29_self_attention(M29_CASH)
raw_mha = multi_head_attention(X_CASH_CONTEXT, params)
print("M29 cash bank weights", m29_cash.weights[0, BANK_INDEX])
print("raw head 0 bank weights", raw_mha.head_weights[0, 0, BANK_INDEX])
print("raw head 1 bank weights", raw_mha.head_weights[0, 1, BANK_INDEX])
print("raw head 0 matches M29", np.allclose(raw_mha.head_weights[0, 0], m29_cash.weights[0]))
print("pre-norm block feeds LN(x), not raw x")
print("block attn_norm bank", cash.attn_norm[0, BANK_INDEX])
print("block head 0 bank weights", cash.head_weights[0, 0, BANK_INDEX])
assert np.allclose(raw_mha.head_weights[0, 0], m29_cash.weights[0])
assert not np.allclose(raw_mha.head_weights[0, 0], raw_mha.head_weights[0, 1])
assert not np.allclose(cash.head_weights[0, 0], raw_mha.head_weights[0, 0])
print("M29 still lives in each head; pre-norm changes the input it sees")


### Pre-norm means M29 does not see raw X

Identity-split head 0 on **raw** X reproduces M29's cash weights.
The teaching block LayerNorms first, so the same M29 function sees
`LN(x)` and the weights move. That is the declared convention, not a
bug. Post-norm would call M29 on raw X and LayerNorm after the add.


## Predict before running — head decomposition

Timestamp a prediction before `run-heads`.

Same cash input and identity parameters. Inspect per-head tensors
**before** merge and `W_O`.

Predict:
- shape of `q_heads` and `head_outputs`
- whether `merge_heads(head_outputs)` equals `attn_concat`
- whether identity `W_O` leaves `attn_projected` equal to concat
- whether either head may be called "the syntax head"

Invariant: input and parameters stay fixed.


In [ ]:
mha = cash.multi_head
print("q_heads", mha.q_heads.shape, "head_outputs", mha.head_outputs.shape)
print("concat", mha.attn_concat.shape, "projected", mha.attn_projected.shape)
print("merge roundtrip", np.allclose(merge_heads(mha.head_outputs), mha.attn_concat))
print("identity W_O projected equals concat", np.allclose(mha.attn_concat, mha.attn_projected))
print("head 0 output\n", mha.head_slice(0)["output"][0])
print("head 1 output\n", mha.head_slice(1)["output"][0])
print(HEAD_INTERPRETATION_LIMIT)
assert mha.q_heads.shape == (1, 3, 2, 2)
assert mha.attn_concat.shape == (1, 3, 4)
assert np.allclose(merge_heads(mha.head_outputs), mha.attn_concat)

half_params = params_with_output_projection(params, TEACHING_W_O_HALF)
half = transformer_block(X_CASH_CONTEXT, half_params)
print("half W_O first divergence", first_divergence(cash, half))
print("half projected is 0.5 concat", np.allclose(half.attn_projected, 0.5 * half.attn_concat))


### Split, merge, then a matrix — still not a personality

Two heads are a reshape of a concatenated projection. Merge inverts
the split. `W_O` is the named change that can mix the concatenated
channels. Identity `W_O` hides that matrix; a half-scale `W_O` makes
it visible at `attn_projected`. Neither head earned a human title.


## Predict before running — residual add

Timestamp a prediction before `run-residual`.

Declared pre-norm. The attention sublayer has already produced
`attn_projected`. The residual identity is `stream + sublayer`.

Predict:
- whether `attn_add` equals `x + attn_projected` or `LN(x) + attn_projected`
- the hand sum `(1, 0, 2) + (0, 1, -1)`
- whether `attn_residual` equals `attn_add` in pre-norm

Do not claim a training-stability theorem. This is an add.


In [ ]:
hand = residual_add(RESIDUAL_STREAM, RESIDUAL_SUBLAYER)
print("hand residual", hand, "expected", RESIDUAL_SUM)
independent = cash.x + cash.attn_projected
print("independent x + attn_projected matches attn_add", np.allclose(independent, cash.attn_add))
print("attn_add equals attn_norm + projected?", np.allclose(cash.attn_add, cash.attn_norm + cash.attn_projected))
print("attn_residual equals attn_add", np.allclose(cash.attn_residual, cash.attn_add))
print(RESIDUAL_INTERPRETATION_LIMIT)
assert np.allclose(hand, RESIDUAL_SUM)
assert np.allclose(cash.attn_add, cash.x + cash.attn_projected)
assert np.allclose(cash.output, cash.attn_residual + cash.ffn_projected)


### Pre-norm residual skips from the original stream

The add used `x`, not `LN(x)`. If someone adds the sublayer to the
normalized branch, the tensor still looks like a hidden state. Named
checkpoints are how you catch that. The FFN residual is the same
identity one sublayer later: `attn_residual + ffn_projected`.


## Predict before running — residual ablation

Timestamp a prediction before `run-ablation`.

Same cash parameters and convention. The named change is
**removing** the attention residual add (`skip_residual="attn"`).

Predict:
- whether `attn_projected` stays equal to the healthy block
- whether `attn_add` still equals `x + attn_projected`
- whether output L2 at bank must move

This is an experiment, not a defect. Parameters stay fixed. Do not
generalize to "residuals always help training."


In [ ]:
skipped = ablate_residual(X_CASH_CONTEXT, params, which="attn")
print("skip", skipped.skip_residual)
print("projected equal", np.allclose(skipped.attn_projected, cash.attn_projected))
print("attn_add equals projected", np.allclose(skipped.attn_add, skipped.attn_projected))
print("first divergence", first_divergence(cash, skipped))
print("healthy stream L2", stream_l2(cash))
print("ablated stream L2", stream_l2(skipped))
assert skipped.skip_residual == "attn"
assert np.allclose(skipped.attn_projected, cash.attn_projected)
assert not np.allclose(skipped.output, cash.output)


### Dropping one add moves the stream without touching W

`attn_projected` was unchanged. The skip replaced `x + sublayer`
with the sublayer alone. Output statistics moved. That is all this
experiment can prove. It cannot prove a training-stability story.


## Predict before running — norm placement

Timestamp a prediction before `run-norm`.

Same cash input and parameters. The named change is the **labeled**
convention: `pre_norm` versus `post_norm`.

Predict:
- whether `attn_norm` still equals `LN(x)` under post-norm
- whether block outputs must match
- whether a difference would prove one convention is universally better

Architecture dependence is the claim. Not a quality ranking.


In [ ]:
post = transformer_block(X_CASH_CONTEXT, params, convention="post_norm")
print("post convention", post.declared_convention, "defect", post.defect)
print("first divergence vs pre", first_divergence(cash, post))
print("post attn_norm equals raw x", np.allclose(post.attn_norm, post.x))
print("pre attn_norm equals LN(x)", np.allclose(cash.attn_norm, layer_norm(cash.x, params.ln1_gamma, params.ln1_beta, params.ln_eps)))
print("post attn_residual equals LN(attn_add)", np.allclose(post.attn_residual, layer_norm(post.attn_add, params.ln1_gamma, params.ln1_beta, params.ln_eps)))
print("outputs equal?", np.allclose(cash.output, post.output))
print(NORM_INTERPRETATION_LIMIT)
assert post.defect == "none"
assert first_divergence(cash, post) == "attn_norm"
assert not np.allclose(cash.output, post.output)


### Placement is a convention, not a vibe

Post-norm runs M29 on raw `x` and LayerNorms **after** each residual
add. Pre-norm LayerNorms **before** each sublayer and leaves the
residual stream unnormalized. Same weights, different graph, different
tensors. Neither run trained anything, so neither is a recipe for a
production LLM.


## Predict before running — feed-forward change

Timestamp a prediction before `run-ffn`.

Same cash input, same attention path. The named change is FFN
activation: ReLU versus identity. Weights stay put.

Predict:
- whether `attn_residual` stays equal
- which checkpoint should be the first to move
- whether the FFN can mix token positions

The FFN is the same MLP at every row.


In [ ]:
identity_ffn = transformer_block(
    X_CASH_CONTEXT, params_with_activation(params, activation="identity")
)
print("attn residual equal", np.allclose(cash.attn_residual, identity_ffn.attn_residual))
print("first divergence", first_divergence(cash, identity_ffn))
print("relu hidden bank", cash.ffn_hidden[0, BANK_INDEX])
print("identity hidden bank", identity_ffn.ffn_hidden[0, BANK_INDEX])
print("hidden width", cash.ffn_hidden.shape[-1], "d_ff", D_FF)
ln_demo = layer_norm(LN_VECTOR)
print("LN(3, 1)", ln_demo, "(eps keeps this off of exactly ±1)")
assert np.allclose(cash.attn_residual, identity_ffn.attn_residual)
assert first_divergence(cash, identity_ffn) == "ffn_hidden"
print("FFN changed features, not the attention residual")


### Position-wise means no new mixing across tokens

Attention already mixed positions. The FFN applies one MLP at each
position. Switching ReLU off changes hidden coordinates that were
negative and leaves `attn_residual` alone. Width `d_ff=8` is an
expand-then-project, not a second sequence mixer.


## Predict before running — reference parity

Timestamp a prediction before `run-parity`.

Compare the teaching block to two trusted fixtures: frozen golden
tensors and `independent_pre_norm_compose` (explicit numpy `+`).

Predict:
- whether `first_divergence` against the golden pre-norm trace is `None`
- whether independent `x + attn_projected` matches `attn_add`
- which checkpoint would move first if the residual used `LN(x)`

Tolerance at every TRACE_CHECKPOINTS boundary is the contract.


In [ ]:
golden = golden_pre_norm_trace()
composed = independent_pre_norm_compose(X_CASH_CONTEXT, params)
print("checkpoints", TRACE_CHECKPOINTS)
print("parity vs golden", checkpoint_parity(cash, golden))
print("compose attn_add matches", np.allclose(cash.attn_add, composed["attn_add"]))
print("compose output matches", np.allclose(cash.output, composed["output"]))
print("frozen attn_residual", np.allclose(cash.attn_residual[0], GOLDEN_CASH_ATTN_RESIDUAL, atol=1e-9))
print("frozen output", np.allclose(cash.output[0], GOLDEN_CASH_OUTPUT, atol=1e-9))
assert checkpoint_parity(cash, golden)["match"]
assert np.allclose(cash.output[0], GOLDEN_CASH_OUTPUT, atol=1e-9)
print("healthy path matches the fixture at every named boundary")


### Named intermediates are the observability surface M31 inherits

If only the output matched, a residual could still be on the wrong
branch. Matching every checkpoint, including the independent `+`,
is what makes the golden fixture a diagnosis tool rather than a
screenshot of `y`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.3))
l2 = stream_l2(cash)
positions = np.arange(3)
axes[0].bar(positions - 0.2, l2["x"], width=0.4, label="x")
axes[0].bar(positions + 0.2, l2["attn_residual"], width=0.4, label="attn residual")
axes[0].set_title("L2: stream before/after attn add")
axes[0].set_xticks(positions, list(CASH_TOKENS))
axes[0].set_ylabel("L2")
axes[0].legend()
im1 = axes[1].imshow(cash.head_weights[0, 0], vmin=0.0, vmax=1.0, cmap="viridis")
axes[1].set_title("head 0 weights (not a role)")
axes[1].set_xlabel("key")
axes[1].set_ylabel("query")
axes[1].set_xticks(range(3), list(CASH_TOKENS))
axes[1].set_yticks(range(3), list(CASH_TOKENS))
fig.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(cash.head_weights[0, 1], vmin=0.0, vmax=1.0, cmap="viridis")
axes[2].set_title("head 1 weights (not a role)")
axes[2].set_xlabel("key")
axes[2].set_xticks(range(3), list(CASH_TOKENS))
axes[2].set_yticks(range(3), list(CASH_TOKENS))
fig.colorbar(im2, ax=axes[2], fraction=0.046)
fig.suptitle("Where the residual stream moved; heads remain unnamed")
fig.tight_layout()
plt.show()
plt.close(fig)
print("output L2", l2["output"], "attn projected L2", l2["attn_projected"])


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `split_heads`, `multi_head_attention`, `transformer_block`,
`residual_add`, `layer_norm`, and `feed_forward` in
`missions/M30/transformer_block.py`.

Predict:
- shapes of `q_heads`, `head_outputs`, `attn_concat`, and `output`
  for cash `(3, 4)` with `n_heads=2`
- whether the pre-norm residual add uses `x` or `attn_norm`
- whether the FFN mixes positions
- which M29 function is called inside `multi_head_attention`

Do not search the file for a training loop or a temperature.


In [ ]:
block_src = inspect.getsource(transformer_block)
mha_src = inspect.getsource(multi_head_attention)
ffn_src = inspect.getsource(feed_forward)
print("calls scaled_dot_product_attention", "scaled_dot_product_attention" in mha_src)
print("block uses residual_add", "residual_add" in block_src)
print("block uses layer_norm", "layer_norm" in block_src)
print("FFN has no sequence-mix matmul over positions", "@" in ffn_src and "transpose" not in ffn_src)
print("repair recomputes from trace.x", "trace.x" in inspect.getsource(repair_block))
print("cash q_heads", cash.shapes["q_heads"], "concat", cash.shapes["attn_concat"], "output", cash.shapes["output"])
print("split 4 into 2x2", split_heads(cash.q, 2).shape)
assert cash.shapes["q_heads"] == (1, 3, 2, 2)
assert cash.shapes["output"] == (1, 3, 4)
assert "scaled_dot_product_attention" in mha_src
print("softmax stays in M29; the block only splits, calls, merges, adds, and norms")


## Predict before running — Controlled failure: residual branch

Timestamp a prediction before `run-failure-residual`.

Cash `x` and parameters stay fixed. The named change is
`defect="residual_wrong_branch"`.

Predict:
- whether `attn_projected` still matches the golden block
- which named checkpoint should be the **first** to move
- whether `attn_add` still equals `x + attn_projected`

Finite hidden states are not proof the skip is attached to the
correct tensor.


In [ ]:
broken_residual = block_with_defect(
    X_CASH_CONTEXT, params, defect="residual_wrong_branch"
)
print("defect", broken_residual.defect)
print("parity", checkpoint_parity(cash, broken_residual))
print("projected equal", np.allclose(broken_residual.attn_projected, cash.attn_projected))
print("attn_add equals x + projected?", np.allclose(broken_residual.attn_add, broken_residual.x + broken_residual.attn_projected))
print("attn_add equals attn_norm + projected?", np.allclose(broken_residual.attn_add, broken_residual.attn_norm + broken_residual.attn_projected))
assert broken_residual.defect == "residual_wrong_branch"
assert first_divergence(cash, broken_residual) == "attn_add"
print("the first named mismatch is the residual add, not the attention call")


## Predict before running — Controlled failure: norm boundary

Timestamp a prediction before `run-failure-norm`.

Same cash `x` and parameters. The named change is
`defect="norm_wrong_boundary"`: the run is still **labeled**
`pre_norm`.

Predict:
- whether `attn_norm` still equals `LN(x)`
- which named checkpoint should move first
- how this differs from a **labeled** `post_norm` run (does the
  residual-stream checkpoint get LayerNormed after the add?)

Do not repair yet. Parity first.


In [ ]:
broken_norm = block_with_defect(
    X_CASH_CONTEXT, params, defect="norm_wrong_boundary"
)
print("declared", broken_norm.declared_convention, "defect", broken_norm.defect)
print("parity", checkpoint_parity(cash, broken_norm))
print("attn_norm equals x?", np.allclose(broken_norm.attn_norm, broken_norm.x))
print("attn_norm equals LN(x)?", np.allclose(broken_norm.attn_norm, cash.attn_norm))
print("residual still x + projected?", np.allclose(broken_norm.attn_add, broken_norm.x + broken_norm.attn_projected))
print("contrast labeled post_norm residual LN after add", not np.allclose(post.attn_residual, post.attn_add))
assert broken_norm.declared_convention == "pre_norm"
assert first_divergence(cash, broken_norm) == "attn_norm"
print("skipping the first LN is not the same graph as labeled post-norm")


### Diagnose before repair

Symptom: both defective blocks emit finite `(1, 3, 4)` tensors.

Hypotheses worth separating: the token table changed; `W_O` changed;
the residual add used the normalized branch; the first LayerNorm was
skipped while the label still said pre-norm; someone silently ran
post-norm.

The discriminating observations are `first_divergence`, whether
`attn_projected` still matches, whether `attn_add` equals
`x + attn_projected`, and whether `attn_residual` was LayerNormed.

The root cause is a skip attachment or a norm boundary, not a missing
trainer. Do not repair this by stacking blocks or adding a loss.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

The repair must start from the **broken trace** (`repair_block`),
not from a second unrelated `defect="none"` call. `x`, parameters,
mask, and declared convention stay the broken objects.

Predict:
- whether repaired checkpoints match the golden cash block
- whether a freshly broken call still diverges (regression)
- whether repair is allowed to change the teaching vectors


In [ ]:
repaired = repair_block(broken_residual)
print("repair first divergence vs cash", first_divergence(cash, repaired))
print("repair uses broken x", np.allclose(repaired.x, broken_residual.x))
print("repair defect", repaired.defect)
repaired_norm = repair_block(broken_norm)
print("norm repair first divergence", first_divergence(cash, repaired_norm))

still_broken = block_with_defect(
    broken_residual.x, broken_residual.params, defect="residual_wrong_branch"
)
print("regression still diverges at", first_divergence(cash, still_broken))
assert first_divergence(cash, repaired) is None
assert first_divergence(cash, repaired_norm) is None
assert first_divergence(cash, still_broken) == "attn_add"
print("pre-norm identities restored from the broken objects")


### Restore the addend and the first LN; leave the tensors

The healthy residuals were a numpy `+` away from tensors the broken
trace already held. Repair recomputed the declared pre-norm graph
from those objects. A second defective call still diverges, which is
the regression. M31 can consume this block; it does not get to
redefine the residual identity.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- cash vs water block outputs under declared pre-norm
- head shapes, merge, and a no-mythology statement
- independent `x + attn_projected` matching `attn_add`
- residual ablation with parameters fixed
- labeled pre-norm versus post-norm
- FFN activation change with attention residual fixed
- golden checkpoint parity
- residual/norm diagnosis and `repair_block`

See `missions/M30/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M30/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Draw the pre-norm block, annotate shapes for a fresh
`B=2, T=5, d_model=8, n_heads=4` configuration, add a residual by
hand, LayerNorm a two-vector, mark a wrong residual placement, and
list which M29 pieces still run inside each head.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M30/adr_prompt.md` to choose a V06 teaching-block
convention (pre vs post, activation, head dimensions, checkpoints,
version identity). Do not claim a production LLM.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M30 for the package, not as a substitute for the learner ADR.


## M29 → M30 handoff

M29 supplied a single-head trace. M30 composes that head into a
block with split/merge, `W_O`, residual adds, a declared pre-norm
convention, and a position-wise FFN.

M31 may attach a next-token objective to stacks of this block. It
must not relabel residuals as a training result without a training
experiment, and it must not treat a head as a linguistic role.

M32 stays closed. There is no decoder here.

Reusable artifacts: `BlockTrace` checkpoints
(`x`, `attn_norm`, heads, `attn_projected`, `attn_add`,
`attn_residual`, `ffn_norm`, `ffn_hidden`, `ffn_projected`,
`output`), version `v06-teaching-block-1`, and the cash/water 4-D
sequences.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why does pre-norm feed M29 `LN(x)` rather than raw `x`?
2. What identity tells you a residual is attached to the correct stream?
3. Why are two different head-weight tables not evidence of two human roles?
4. What must M31 receive that a block output tensor without named
   checkpoints cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert cash.declared_convention == "pre_norm"
assert np.allclose(raw_mha.head_weights[0, 0], m29_cash.weights[0])
assert np.allclose(cash.attn_add, cash.x + cash.attn_projected)
assert np.allclose(hand, RESIDUAL_SUM)
assert first_divergence(cash, skipped) == "attn_add"
assert first_divergence(cash, post) == "attn_norm"
assert first_divergence(cash, identity_ffn) == "ffn_hidden"
assert checkpoint_parity(cash, golden)["match"]
assert np.allclose(cash.output[0], GOLDEN_CASH_OUTPUT, atol=1e-9)
assert first_divergence(cash, broken_residual) == "attn_add"
assert first_divergence(cash, broken_norm) == "attn_norm"
assert first_divergence(cash, repaired) is None
assert first_divergence(cash, still_broken) == "attn_add"
assert "parallel learned projections" in HEAD_INTERPRETATION_LIMIT
assert "stream + sublayer" in RESIDUAL_INTERPRETATION_LIMIT
print("M30 integrity checks passed")
